This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [1]:
!pip install keras keras-hub --upgrade -q

In [2]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [3]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## Language models and the Transformer

### The language model

#### Training a Shakespeare language model

In [4]:
import keras

filename = keras.utils.get_file(
    origin=(
        "https://storage.googleapis.com/download.tensorflow.org/"
        "data/shakespeare.txt"
    ),
)
shakespeare = open(filename, "r").read()

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
shakespeare[:250]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n'

In [6]:
import tensorflow as tf

sequence_length = 100

def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]

features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

In [7]:
x, y = next(dataset.as_numpy_iterator())
x[:50], y[:50]

(b'First Citizen:\nBefore we proceed any further, hear',
 b'irst Citizen:\nBefore we proceed any further, hear ')

In [8]:
from keras import layers

tokenizer = layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))

In [9]:
vocabulary_size = tokenizer.vocabulary_size()
vocabulary_size

67

In [10]:
dataset = dataset.map(
    lambda features, labels: (tokenizer(features), tokenizer(labels)),
    num_parallel_calls=8,
)
training_data = dataset.shuffle(10_000).batch(64).cache()

In [11]:
embedding_dim = 256
hidden_dim = 1024

inputs = layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = layers.GRU(hidden_dim, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)

In [12]:
model.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)            │ (None, 100)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ embedding (Embedding)             │ (None, 100, 256)         │        17,152 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ gru (GRU)                         │ (None, 100, 1024)        │     3,938,304 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 100, 1024)        │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 100, 67)          │        68,675 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 4,024,131 (15.35 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data, epochs=20)

Epoch 1/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 20s 82ms/step - loss: 2.7238 - sparse_categorical_accuracy: 0.2854
Epoch 2/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 68ms/step - loss: 2.0035 - sparse_categorical_accuracy: 0.4138
Epoch 3/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 69ms/step - loss: 1.7381 - sparse_categorical_accuracy: 0.4833
Epoch 4/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 70ms/step - loss: 1.5857 - sparse_categorical_accuracy: 0.5241
Epoch 5/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 71ms/step - loss: 1.4919 - sparse_categorical_accuracy: 0.5489
Epoch 6/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 73ms/step - loss: 1.4272 - sparse_categorical_accuracy: 0.5655
Epoch 7/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - loss: 1.3760 - sparse_categorical_accuracy: 0.5784
Epoch 8/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 77ms/step - loss: 1.3331 - sparse_categorical_accuracy: 0.5892
Epoch 9/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 80ms/step - loss: 1.2942 - sparse_categorical_accuracy: 0.5995
Epoch 10/20
175/175 ━━━━━━━━━━━━━━━━━

#### Generating Shakespeare

In [14]:
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")

x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = layers.GRU(hidden_dim, return_state=True)(
    x, initial_state=input_state
)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model = keras.Model(
    inputs=(inputs, input_state),
    outputs=(outputs, output_state),
)
generation_model.set_weights(model.get_weights())

In [15]:
tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))

prompt = """
Oh, what is our mission in the age of AI!
"""

In [16]:
input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

In [17]:
import numpy as np

generated_ids = []
max_length = 250
for i in range(max_length):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

In [18]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)


Oh, what is our mission in the age of AI!
Married the thing you well; and I could fear
The standers of your daughters; then to give thee thee,
And thou a thousand than a wild enforced to the sun:
So many heavy son and be as thought
To hear him seem'd to hear the beast of his grave;
And thou 


### Sequence-to-sequence learning

#### English-to-Spanish translation

In [19]:
import pathlib

zip_path = keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True,
)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [20]:
with open(text_path) as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    # english, spanish = line.split("\t")
    # spanish = "[start] " + spanish + " [end]"
    # text_pairs.append((english, spanish))

    english, spanish = line.split("\t")
    english = "[start] " + english + " [end]"
    text_pairs.append((spanish, english))

In [21]:
import random
random.choice(text_pairs)

('Su vestido parecía barato.', '[start] Her dress looked cheap. [end]')

In [22]:
import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

In [23]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 15000
sequence_length = 20

# english_tokenizer = layers.TextVectorization(
#     max_tokens=vocab_size,
#     output_mode="int",
#     output_sequence_length=sequence_length,
# )
# spanish_tokenizer = layers.TextVectorization(
#     max_tokens=vocab_size,
#     output_mode="int",
#     output_sequence_length=sequence_length + 1,
#     standardize=custom_standardization,
# )
# train_english_texts = [pair[0] for pair in train_pairs]
# train_spanish_texts = [pair[1] for pair in train_pairs]
# english_tokenizer.adapt(train_english_texts)
# spanish_tokenizer.adapt(train_spanish_texts)



spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_spanish_texts = [pair[0] for pair in train_pairs]
train_english_texts = [pair[1] for pair in train_pairs]
spanish_tokenizer.adapt(train_spanish_texts)
english_tokenizer.adapt(train_english_texts)

In [53]:
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    # features = {"english": eng, "spanish": spa[:, :-1]}
    # labels = spa[:, 1:]

    # Change "english" and "spanish" input names to match new roles
    features = {"spanish": spa, "english": eng[:, :-1]}
    labels = eng[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    spa_texts, eng_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [54]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)

(64, 20)


In [55]:
print(inputs["spanish"].shape)

(64, 20)


In [56]:
print(targets.shape)

(64, 20)


In [58]:
print(sample_weights.shape)

(64, 20)


#### Sequence-to-sequence learning with RNNs

In [59]:
embed_dim = 256
hidden_dim = 1024

#source = keras.Input(shape=(None,), dtype="int32", name="english")
source = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)

In [60]:
#target = keras.Input(shape=(None,), dtype="int32", name="spanish")
target = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
rnn_layer = layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, target], target_predictions)

In [61]:
seq2seq_rnn.summary(line_length=80)

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_10          │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_6           │ (None, None)      │           0 │ spanish[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_11          │ (None, None, 256) │   3,840,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional_1       │ (None, 1024)      │   7,876,608 │ embedding_10[0][0… │
│ (Bidirectional)       │                   │             │ not_equal_6[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ gru_5 (GRU)           │ (None, None,      │   3,938,304 │ embedding_11[0][0… │
│                       │ 1024)             │             │ bidirectional_1[0… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_24 (Dropout)  │ (None, None,      │           0 │ gru_5[0][0]        │
│                       │ 1024)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_15 (Dense)      │ (None, None,      │  15,375,000 │ dropout_24[0][0]   │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 34,869,912 (133.02 MB)

 Trainable params: 34,869,912 (133.02 MB)

 Non-trainable params: 0 (0.00 B)

In [62]:
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 131s 97ms/step - accuracy: 0.3606 - loss: 3.5401 - val_accuracy: 0.5068 - val_loss: 2.4837
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 120s 92ms/step - accuracy: 0.5516 - loss: 2.1571 - val_accuracy: 0.6183 - val_loss: 1.8446
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 120s 92ms/step - accuracy: 0.6464 - loss: 1.5390 - val_accuracy: 0.6645 - val_loss: 1.5931
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 120s 92ms/step - accuracy: 0.7069 - loss: 1.1659 - val_accuracy: 0.6853 - val_loss: 1.5018
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 120s 92ms/step - accuracy: 0.7530 - loss: 0.9158 - val_accuracy: 0.6968 - val_loss: 1.4719
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 120s 92ms/step - accuracy: 0.7871 - loss: 0.7484 - val_accuracy: 0.7027 - val_loss: 1.4718
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 120s 92ms/step - accuracy: 0.8114 - loss: 0.6383 - val_accuracy: 0.7054 - val_loss: 1.4979
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 120s 92ms/step - accuracy: 

In [63]:
import numpy as np

#spa_vocab = spanish_tokenizer.get_vocabulary()
spa_vocab = english_tokenizer.get_vocabulary()

#spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
eng_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = spanish_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        #tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence = english_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        #sampled_token = spa_index_lookup[sampled_token_index]
        sampled_token = eng_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
La conozco bien.
[start] i know her well [end]
-
Este té está ardiendo.
[start] this tea is sour [end]
-
Te llamé hace tres horas.
[start] i called you up three hours [end]
-
Es un buen sistema.
[start] its a good system [end]
-
¿Le diste a Tom algo de comer?
[start] did you give tom anything to eat [end]


### The Transformer architecture

#### Dot-product attention

#### Transformer encoder block

In [64]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

#### Transformer decoder block

In [65]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

#### Sequence-to-sequence learning with a Transformer

In [66]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

#source = keras.Input(shape=(None,), dtype="int32", name="english")
source = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

#target = keras.Input(shape=(None,), dtype="int32", name="spanish")
target = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

In [67]:
transformer.summary(line_length=80)

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_12          │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_8           │ (None, None)      │           0 │ spanish[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_encoder_2 │ (None, None, 256) │   1,315,072 │ embedding_12[0][0… │
│ (TransformerEncoder)  │                   │             │ not_equal_8[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_9           │ (None, None)      │           0 │ spanish[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_13          │ (None, None, 256) │   3,840,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_decoder_2 │ (None, None, 256) │   1,578,752 │ transformer_encod… │
│ (TransformerDecoder)  │                   │             │ not_equal_9[0][0], │
│                       │                   │             │ embedding_13[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_28 (Dropout)  │ (None, None, 256) │           0 │ transformer_decod… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_20 (Dense)      │ (None, None,      │   3,855,000 │ dropout_28[0][0]   │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 14,428,824 (55.04 MB)

 Trainable params: 14,428,824 (55.04 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.4241 - loss: 1.3466 - val_accuracy: 0.5674 - val_loss: 0.9219
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.6022 - loss: 0.8290 - val_accuracy: 0.6356 - val_loss: 0.7492
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.6712 - loss: 0.6340 - val_accuracy: 0.6595 - val_loss: 0.6949
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.7131 - loss: 0.5199 - val_accuracy: 0.6738 - val_loss: 0.6754
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.7437 - loss: 0.4418 - val_accuracy: 0.6777 - val_loss: 0.6714
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.7671 - loss: 0.3846 - val_accuracy: 0.6806 - val_loss: 0.6860
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.7883 - loss: 0.3403 - val_accuracy: 0.6850 - val_loss: 0.6878
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.8041 -

#### Embedding positional information

In [69]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

In [70]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

#source = keras.Input(shape=(None,), dtype="int32", name="english")
source = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

#target = keras.Input(shape=(None,), dtype="int32", name="spanish")
target = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

In [71]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.4328 - loss: 1.3215 - val_accuracy: 0.5993 - val_loss: 0.8678
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.6339 - loss: 0.7755 - val_accuracy: 0.6741 - val_loss: 0.6842
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.7001 - loss: 0.5888 - val_accuracy: 0.7012 - val_loss: 0.6216
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 42s 24ms/step - accuracy: 0.7381 - loss: 0.4846 - val_accuracy: 0.7147 - val_loss: 0.5957
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.7650 - loss: 0.4151 - val_accuracy: 0.7196 - val_loss: 0.5901
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.7859 - loss: 0.3630 - val_accuracy: 0.7251 - val_loss: 0.5880
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.8045 - loss: 0.3227 - val_accuracy: 0.7279 - val_loss: 0.5984
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.8180 -

In [72]:
import numpy as np

spa_vocab = english_tokenizer.get_vocabulary()
eng_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = spanish_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = english_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = transformer.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = eng_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(10):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
Estoy justo detrás de ti.
[start] im right behind you [end]
-
No tengo prisa.
[start] im in a hurry [end]
-
Era pobre y vivía en una pequeña cabaña.
[start] he was poor and lived in a small cabin [end]
-
Tu pálpito sobre Tom era correcto.
[start] your tide about tom was right [end]
-
Tom llega temprano.
[start] tom arrives early [end]
-
Tom, me gustaría que conozcas a alguien.
[start] tom wish you meet somebody [end]
-
Tom es el primo de Mary.
[start] tom is marys cousin [end]
-
¡Habla conmigo!
[start] wake me up [end]
-
Pocos estudiantes entienden latín.
[start] few students taste smoke [end]
-
Procuraré estar allí para las dos y media.
[start] you let me be there by 230 [end]


The below is not needed for the scope of this problem

### Classification with a pretrained Transformer

#### Pretraining a Transformer encoder

#### Loading a pretrained Transformer

In [73]:
import keras_hub

tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")
backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")

In [74]:
tokenizer("The quick brown fox")

Array([  133,  2119,  6219, 23602], dtype=int32)

In [75]:
backbone.summary(line_length=80)

Model: "roberta_backbone"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ token_ids             │ (None, None)      │           0 │ -                  │
│ (InputLayer)          │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embeddings            │ (None, None, 768) │  38,996,736 │ token_ids[0][0]    │
│ (TokenAndPositionEmb… │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embeddings_layer_norm │ (None, None, 768) │       1,536 │ embeddings[0][0]   │
│ (LayerNormalization)  │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embeddings_dropout    │ (None, None, 768) │           0 │ embeddings_layer_… │
│ (Dropout)             │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ padding_mask          │ (None, None)      │           0 │ -                  │
│ (InputLayer)          │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_0   │ (None, None, 768) │   7,087,872 │ embeddings_dropou… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_1   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_2   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_3   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_4   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_5   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_6   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_7   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_8   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_9   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_10  │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │      

 Total params: 124,052,736 (473.22 MB)

 Trainable params: 124,052,736 (473.22 MB)

 Non-trainable params: 0 (0.00 B)

#### Preprocessing IMDb movie reviews

In [76]:
import os, pathlib, shutil, random

zip_path = keras.utils.get_file(
    origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    fname="imdb",
    extract=True,
)

imdb_extract_dir = pathlib.Path(zip_path) / "aclImdb"
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

shutil.copytree(imdb_extract_dir / "test", test_dir, dirs_exist_ok=True)

val_percentage = 0.2
for category in ("neg", "pos"):
    src_dir = imdb_extract_dir / "train" / category
    src_files = os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val_samples = int(len(src_files) * val_percentage)

    os.makedirs(train_dir / category, exist_ok=True)
    os.makedirs(val_dir / category, exist_ok=True)
    for index, file in enumerate(src_files):
        if index < num_val_samples:
            shutil.copy(src_dir / file, val_dir / category / file)
        else:
            shutil.copy(src_dir / file, train_dir / category / file)

In [77]:
from keras.utils import text_dataset_from_directory

batch_size = 16
train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = text_dataset_from_directory(test_dir, batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [86]:
def preprocess(text, label):
    packer = keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        return_padding_mask=True,
    )
    token_ids, padding_mask = packer(tokenizer(text))
    return {"token_ids": token_ids, "padding_mask": padding_mask}, label

preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds = val_ds.map(preprocess)
preprocessed_test_ds = test_ds.map(preprocess)

In [85]:
next(iter(preprocessed_train_ds))

({'token_ids': <tf.Tensor: shape=(16, 512), dtype=int32, numpy=
  array([[    0,  3609,  4344, ...,     1,     1,     1],
         [    0, 36369,    18, ...,     1,     1,     1],
         [    0,   713,    16, ...,     1,     1,     1],
         ...,
         [    0,   100,   794, ...,     1,     1,     1],
         [    0,   605, 18308, ...,     1,     1,     1],
         [    0,   100,   348, ...,     1,     1,     1]], dtype=int32)>,
  'padding_mask': <tf.Tensor: shape=(16, 512), dtype=bool, numpy=
  array([[ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         ...,
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False]])>},
 <tf.Tensor: shape=(16,), dtype=int32, numpy=array([0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0], dtype=int32)>)

#### Fine-tuning a pretrained Transformer

In [84]:
inputs = backbone.input
x = backbone(inputs)
x = x[:, 0, :]
x = layers.Dropout(0.1)(x)
x = layers.Dense(768, activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
classifier = keras.Model(inputs, outputs)

In [87]:
classifier.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
classifier.fit(
    preprocessed_train_ds,
    validation_data=preprocessed_val_ds,
)

RuntimeError: Unable to automatically build the model. Please build it yourself before calling fit/evaluate/predict. A model is 'built' when its variables have been created and its `self.built` attribute is True. Usually, calling the model on a batch of data is the right way to build it.
Exception encountered:
'Exception encountered when calling ReversibleEmbedding.call().

[1m'NoneType' object is not callable[0m

Arguments received by ReversibleEmbedding.call():
  • inputs=jnp.ndarray(shape=(16, 512), dtype=int32)
  • reverse=False'

In [83]:
classifier.evaluate(preprocessed_test_ds)

RuntimeError: Unable to automatically build the model. Please build it yourself before calling fit/evaluate/predict. A model is 'built' when its variables have been created and its `self.built` attribute is True. Usually, calling the model on a batch of data is the right way to build it.
Exception encountered:
'Exception encountered when calling ReversibleEmbedding.call().

[1m'NoneType' object is not callable[0m

Arguments received by ReversibleEmbedding.call():
  • inputs=jnp.ndarray(shape=(16, 512), dtype=int32)
  • reverse=False'

### What makes the Transformer effective?